## Portfolio 1

Wij hebben als spel 4 op een rij gekozen.

In [1]:
import sys
!"{sys.executable}" -m pip install pettingzoo chess pygame numpy matplotlib ipykernel

  Using cached pettingzoo-1.25.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached chess-1.11.2-py3-none-any.whl
  Using cached pygame-2.6.1-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached numpy-2.4.3-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached gymnasium-1.2.3-py3-none-any.whl.metadata (10 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp311-cp311-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.1.1-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached Farama_Notifications-0.0.4-py3-none-any.whl.metadata (558 bytes)
Using cached pettingzoo-1.

In [5]:
from pettingzoo.classic import connect_four_v3

In [7]:
env = connect_four_v3.env(render_mode="human")
env.reset()

for agent in env.agent_iter():

    observation, reward, termination, truncation, info = env.last()

    if termination or truncation:
        action = None
    else:
        mask = observation["action_mask"]

        # kies eerste geldige zet
        legal_moves = [i for i, m in enumerate(mask) if m == 1]
        action = legal_moves[0]

    env.step(action)
    

In [22]:
import numpy as np
import random

class BlockingAgent:

    def act(self, observation):

        board = observation["observation"]
        mask = observation["action_mask"]

        legal_moves = [i for i, m in enumerate(mask) if m == 1]

        opponent = board[:, :, 1]

        # 🔥 check elke kolom: voorkomt directe winst
        for col in legal_moves:

            row = self.get_next_open_row(opponent, col)
            if row is None:
                continue

            temp_board = opponent.copy()
            temp_board[row][col] = 1

            if self.check_win(temp_board):
                print("BLOCKING WIN:", col)
                return col

        # 🔥 extra: detecteer 3-op-een-rij dreigingen
        for col in legal_moves:

            row = self.get_next_open_row(opponent, col)
            if row is None:
                continue

            if self.creates_threat(opponent, row, col):
                print("BLOCKING THREAT:", col)
                return col

        return random.choice(legal_moves)

    def get_next_open_row(self, board, col):
        for row in reversed(range(6)):
            if board[row][col] == 0:
                return row
        return None

    def check_win(self, board):

        # horizontaal
        for r in range(6):
            for c in range(4):
                if all(board[r][c+i] == 1 for i in range(4)):
                    return True

        # verticaal
        for r in range(3):
            for c in range(7):
                if all(board[r+i][c] == 1 for i in range(4)):
                    return True

        # diagonaal \
        for r in range(3):
            for c in range(4):
                if all(board[r+i][c+i] == 1 for i in range(4)):
                    return True

        # diagonaal /
        for r in range(3, 6):
            for c in range(4):
                if all(board[r-i][c+i] == 1 for i in range(4)):
                    return True

        return False

    def creates_threat(self, board, row, col):

        temp = board.copy()
        temp[row][col] = 1

        # tel aantal 3-op-een-rij
        count = 0

        # horizontaal
        for c in range(max(0, col-3), min(4, col+1)):
            window = temp[row][c:c+4]
            if np.sum(window) == 3:
                count += 1

        # verticaal
        for r in range(max(0, row-3), min(3, row+1)):
            window = [temp[r+i][col] for i in range(4)]
            if sum(window) == 3:
                count += 1

        # diagonaal \
        for i in range(-3, 1):
            coords = [(row+i+j, col+i+j) for j in range(4)]
            if all(0 <= r < 6 and 0 <= c < 7 for r, c in coords):
                window = [temp[r][c] for r, c in coords]
                if sum(window) == 3:
                    count += 1

        # diagonaal /
        for i in range(-3, 1):
            coords = [(row-i-j, col+i+j) for j in range(4)]
            if all(0 <= r < 6 and 0 <= c < 7 for r, c in coords):
                window = [temp[r][c] for r, c in coords]
                if sum(window) == 3:
                    count += 1

        return count > 0

In [23]:
from pettingzoo.classic import connect_four_v3
#from agents.blocking_agent import BlockingAgent

env = connect_four_v3.env(render_mode="human")
env.reset()

agent_ai = BlockingAgent()

for agent in env.agent_iter():

    observation, reward, termination, truncation, info = env.last()

    if termination or truncation:
        action = None
    else:
        mask = observation["action_mask"]
        legal_moves = [i for i, m in enumerate(mask) if m == 1]

        if agent == "player_0":
            # 👤 JIJ speelt
            print("\nJouw beurt!")
            print("Kies kolom (0-6): ", legal_moves)

            while True:
                try:
                    action = int(input("Jouw zet: "))
                    if action in legal_moves:
                        break
                    else:
                        print("❌ Ongeldige zet, kies uit:", legal_moves)
                except:
                    print("❌ Voer een getal in")

        else:
            # 🤖 Blocking Agent
            action = agent_ai.act(observation)
            print(f"AI speelt kolom: {action}")

    env.step(action)

env.close()


Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
AI speelt kolom: 2

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
AI speelt kolom: 5

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
BLOCKING THREAT: 2
AI speelt kolom: 2

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
BLOCKING THREAT: 2
AI speelt kolom: 2

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
BLOCKING WIN: 2
AI speelt kolom: 2

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
❌ Voer een getal in
BLOCKING THREAT: 5
AI speelt kolom: 5

Jouw beurt!
Kies kolom (0-6):  [0, 1, 3, 4, 5, 6]
BLOCKING WIN: 5
AI speelt kolom: 5

Jouw beurt!
Kies kolom (0-6):  [0, 1, 3, 4, 5, 6]
BLOCKING WIN: 5
AI speelt kolom: 5

Jouw beurt!
Kies kolom (0-6):  [0, 1, 3, 4, 5, 6]
❌ Ongeldige zet, kies uit: [0, 1, 3, 4, 5, 6]
BLOCKING WIN: 5
AI speelt kolom: 5

Jouw beurt!
Kies kolom (0-6):  [0, 1, 3, 4, 6]
BLOCKING WIN: 3
AI speelt kolom: 3

Jouw beurt!
Kies kolom (0-6):  [0, 1, 3, 4, 6]
BLOCKING WIN: 0
AI speelt kolom: 0